# Advanced Agentic Patterns — A Practical Guide

LangGraph 1.x and local Qwen (`qwen3.5:4b`). Eight patterns: what each one is, a small graph you can run, and when you would use it.

**You already know:** `ChatOllama` and a `StateGraph` from Module 15.

**You will be able to:** call tools in a loop, force a schema, critique a draft, follow a plan, pause for a person, route to a specialist, remember a fact in a new chat, and decide whether a question needs a knowledge base.

**You do not need** the PDF index from notebooks 1–3 in this folder. Those notebooks are the full document desk. This one uses toy tools so each pattern runs on its own.

## Outline

| # | Pattern | One-line idea | When it earns its place |
|---|---------|---------------|-------------------------|
| 1 | ReAct agent | Reason, act, observe, repeat | The answer needs a tool |
| 2 | Structured output | Fill a schema instead of returning prose | Code will read the fields |
| 3 | Reflection | Draft, critique, revise | Quality matters more than the first reply |
| 4 | Plan-and-execute | Write the steps, then do each one | The task has an order it must keep |
| 5 | Human-in-the-loop | Pause before a risky action | The action is hard to undo |
| 6 | Supervisor | A coordinator picks a specialist | One job spans different skills |
| 7 | Long-term memory | Facts that survive a new chat | The assistant should know the user |
| 8 | Agentic RAG | The agent decides whether to retrieve | Only some questions need your documents |

Each pattern has the same four parts: theory, a picture, code, and a scenario with something to try.

## Before you run

1. Select the kernel **RAG Env (Python 3.11)**.
2. Ollama is running, and `qwen3.5:4b` is pulled (`ollama pull qwen3.5:4b`).
3. Run the setup cells once. After that you can start at any pattern.
4. Thinking is off, so a pattern is usually under a minute. Pattern 3 calls the model several times.

Install from `requirements.txt` in a terminal if a package is missing. Do not `pip install` inside this notebook. An unpinned install can change the shared class environment.


---
## Setup

These cells check the environment, import the libraries, and create one `llm`. Every pattern below reuses it.


In [1]:
# ── Environment check ─────────────────────────────────────────────────────────
import importlib.util

required = ["langgraph", "langchain", "langchain_ollama", "pydantic", "ollama"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise SystemExit(
        "Missing packages: " + ", ".join(missing) + "\n"
        "Select the RAG Env (Python 3.11) kernel.\n"
        "From this folder, in a terminal: pip install -r requirements.txt"
    )
for name in required:
    print("ok", name)


ok langgraph
ok langchain
ok langchain_ollama
ok pydantic
ok ollama


In [2]:
# ── Core imports ──────────────────────────────────────────────────────────────
from typing import Literal, TypedDict
from uuid import uuid4

import ollama
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, MessagesState, START, StateGraph
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command, interrupt

print("imports ok")


imports ok


In [3]:
# ── Model ─────────────────────────────────────────────────────────────────────
# qwen3.5 can call tools. reasoning=False skips the thinking trace so a live
# class stays fast and .content is the answer, not a <think> block.
# seed and temperature 0 keep a re-run close to the output saved in this notebook.
MODEL = "qwen3.5:4b"

try:
    available = [m.model for m in ollama.list().models]
except Exception as exc:
    raise SystemExit(
        f"Ollama is not reachable ({exc}). Start it with: ollama serve"
    ) from exc

print("Pulled:", ", ".join(available) if available else "(none)")
if MODEL not in available:
    for fallback in ("qwen3.5:9b", "qwen3.5:4b"):
        if fallback in available:
            print(f"{MODEL} is not pulled. Using {fallback}.")
            MODEL = fallback
            break
    else:
        raise SystemExit(
            f"Pull the class model first:\n  ollama pull qwen3.5:4b\n"
            f"Pulled now: {available}"
        )

llm = ChatOllama(
    model=MODEL,
    temperature=0,
    seed=7,
    num_ctx=4096,
    reasoning=False,
    num_predict=320,
)
probe = llm.invoke("Reply with exactly: Agent ready").content.strip()
print(f"Active model: {MODEL}")
print(f"Probe: {probe[:80]}")


Pulled: qwen3.5:0.8b, qwen3.5:4b, qwen3-embedding:4b, qwen3.5:9b, qwen3-embedding:0.6b


Active model: qwen3.5:4b
Probe: Agent ready


### Shared tools

Several patterns need something to call. These two tools are pure functions: arithmetic through a whitelist, and a word count. Later patterns reuse them.


In [4]:
# ── Shared tools ──────────────────────────────────────────────────────────────

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, for example '144 / 12' or '(23 * 7) + 4'.
    Use this for every calculation.
    """
    import ast
    import operator as op

    # Walk the tree and allow only arithmetic. Do not replace this with eval().
    ops = {
        ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
        ast.Div: op.truediv, ast.Pow: op.pow, ast.Mod: op.mod,
        ast.USub: op.neg,
    }

    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return ops[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return ops[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")

    try:
        return str(ev(ast.parse(expression, mode="eval").body))
    except Exception as exc:
        return f"Error: {exc}"


@tool
def word_count(text: str) -> int:
    """Count the words in a piece of text."""
    return len(text.split())


print("Tools:", calculator.name, "|", word_count.name)
print("Test:", calculator.invoke({"expression": "144 / 12 + 1"}))


Tools: calculator | word_count
Test: 13.0


---
# Pattern 1 — ReAct agent

## Theory

ReAct means the model **reasons and acts in a loop**. It does not have to answer in one shot.

```
Thought      I need 144 / 12
Action       calculator("144 / 12")
Observation  12.0
Answer       The result is 12
```

A chat model invents arithmetic and dates. A tool returns the value, and a bad tool result is visible, so the model can try again.

The hand-built version of this loop is a `StateGraph`: a model node, a `ToolNode`, and `tools_condition`. Module 18 builds that wiring so you can see every edge. `create_agent` is the same loop in one call. Older posts import `create_react_agent` from `langgraph.prebuilt`. That function is deprecated. Use `create_agent`.


## How it works

```
user message
     │
     ▼
┌──────────┐
│  model   │
└────┬─────┘
     │
     ├── tool call ──▶ run tool ──▶ observation ──┐
     │                                            │
     │◀───────────────────────────────────────────┘
     │
     └── no tool call
            │
            ▼
       final answer
```


In [5]:
# ── Pattern 1: one ReAct agent ────────────────────────────────────────────────

react_agent = create_agent(
    model=llm,
    tools=[calculator, word_count],
    system_prompt=(
        "You are a helpful assistant. Use the calculator for every calculation "
        "and word_count whenever the user asks how many words are in a phrase."
    ),
)


def run_react(question: str):
    result = react_agent.invoke({"messages": [HumanMessage(question)]})
    print(f"Q: {question}\n")
    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print(f"   action: {call['name']}({call['args']})")
        elif isinstance(message, ToolMessage):
            print(f"   observation: {message.content}")
    print(f"\nAnswer: {result['messages'][-1].content}")


run_react(
    "What is 144 divided by 12, and how many words are in "
    "the phrase 'machine learning is fun'?"
)


Q: What is 144 divided by 12, and how many words are in the phrase 'machine learning is fun'?

   action: calculator({'expression': '144 / 12'})
   action: word_count({'text': 'machine learning is fun'})
   observation: 12.0
   observation: 4

Answer: The result of 144 divided by 12 is **12**. The phrase "machine learning is fun" contains **4** words.


## When to use it

Use ReAct when the answer depends on an action: a calculation, a lookup, an API call, a database query.

A support bot asked "what is $245 plus 8% tax?" should call `calculator`. The tax comes from the tool result.

**You should see** at least one `action` line before the answer. If the answer appears with no action, the model skipped the tool. The system prompt and the tool docstring are what tell it to call.

**Try it.** Add a tool `reverse_string(text)` and ask: *Reverse the word 'agent' and tell me how many letters it has.* The trace should show two tool calls.


---
# Pattern 2 — Structured output

## Theory

A chat model returns prose. A program needs fields it can trust: `card.difficulty`, not a sentence you scrape with a regex.

`with_structured_output(Schema)` asks the model to fill a Pydantic model. On `qwen3.5:4b` the call that works is:

```python
llm.with_structured_output(Schema, method="function_calling")
```

`function_calling` exposes the schema as a tool. The arguments of that tool call are the fields, and Pydantic validates them.

The default method is `json_schema` (Ollama's structured-output API). On this 4B model that request comes back as markdown, and LangChain raises `OutputParserException`. `json_mode` does the same. Module 15 uses `function_calling` for the same reason.


In [6]:
# ── Pattern 2: a validated object ─────────────────────────────────────────────

class ConceptCard(BaseModel):
    """A study card for one ML term."""
    term: str = Field(description="the ML term being explained")
    definition: str = Field(description="one clear sentence")
    analogy: str = Field(description="a simple real-world analogy")
    difficulty: Literal["beginner", "intermediate", "advanced"] = Field(
        description="how hard the concept is"
    )


card_maker = llm.with_structured_output(ConceptCard, method="function_calling")
card = card_maker.invoke("Explain the concept of overfitting as a study card.")

print(type(card).__name__)
print(f"  term       : {card.term}")
print(f"  definition : {card.definition}")
print(f"  analogy    : {card.analogy}")
print(f"  difficulty : {card.difficulty}")
print()
print(card.model_dump_json(indent=2))


ConceptCard
  term       : Overfitting
  definition : A situation where a model learns the training data too well, including its noise and random fluctuations, resulting in poor performance on new, unseen data.
  analogy    : Like memorizing all the answers to your teacher's past test questions instead of actually learning the subject so you can't answer any future questions correctly.
  difficulty : intermediate

{
  "term": "Overfitting",
  "definition": "A situation where a model learns the training data too well, including its noise and random fluctuations, resulting in poor performance on new, unseen data.",
  "analogy": "Like memorizing all the answers to your teacher's past test questions instead of actually learning the subject so you can't answer any future questions correctly.",
  "difficulty": "intermediate"
}


## When to use it

Use structured output when the next reader is code: an API response, a database row, a branch in a graph.

Extracting 10,000 resumes is this pattern. Every row has `name`, `years_experience`, `skills`, and `seniority`, so it can be inserted without a parser.

**You should see** a `ConceptCard`, then JSON with those four keys. `difficulty` is one of the three allowed strings.

**Try it.** The next cell is a complete starter for a review label. Change the review sentence, then run the cell again.


In [7]:
# ── Pattern 2 exercise starter ────────────────────────────────────────────────

class SentimentResult(BaseModel):
    """Label one short review."""
    label: Literal["positive", "negative", "neutral"]
    confidence: float = Field(description="how sure you are, from 0 to 1")
    key_phrase: str = Field(description="the few words that decided the label")


review = "The course was okay but the pace was too fast."
sentiment = llm.with_structured_output(SentimentResult, method="function_calling")
result = sentiment.invoke(review)
print(review)
print(result.label, result.confidence)
print(result.key_phrase)


The course was okay but the pace was too fast.
neutral 0.85
okay, too fast


---
# Pattern 3 — Reflection

## Theory

The first draft is rarely the one you want. Reflection adds a critic. The same model wears two prompts: author, then reviewer. The reviewer either accepts the draft or sends specific changes back.

```
generate ──▶ reflect ──▶ generate ──▶ reflect ──▶ ... ──▶ stop
```

Stop when the review accepts the draft, or after a fixed number of drafts. Without the cap, a strict critic loops until you stop the kernel.

This is the same shape as generate, run tests, read the failure, fix. Here the "test" is a prompt.


In [8]:
# ── Pattern 3: generate, reflect, revise ──────────────────────────────────────

class ReflectState(TypedDict):
    task: str
    draft: str
    critique: str
    iteration: int


def _accepted(critique: str) -> bool:
    # The first line must be the word APPROVED.
    # "APPROVED" in text is the wrong test: it also matches "NOT APPROVED".
    first = critique.strip().splitlines()[0].strip().upper().strip(" .!:")
    return first == "APPROVED"


def generate(state: ReflectState):
    if state.get("critique"):
        prompt = (
            f"Rewrite the explanation of '{state['task']}'.\n\n"
            f"Previous draft:\n{state['draft']}\n\n"
            f"Address this review:\n{state['critique']}\n\n"
            "Write 3 or 4 sentences."
        )
    else:
        # A thin first draft gives the reviewer something real to reject,
        # so the class can see the loop. A strong first draft can be accepted immediately.
        prompt = (
            f"In one short sentence, explain: {state['task']}. "
            "Do not mention gradients, speed, or other activation functions."
        )
    draft = llm.invoke(prompt).content.strip()
    iteration = state.get("iteration", 0) + 1
    print(f"\n--- draft {iteration} ---\n{draft}\n")
    return {"draft": draft, "iteration": iteration}


def reflect(state: ReflectState):
    prompt = (
        f"You are a strict ML instructor reviewing an explanation of '{state['task']}'. "
        "If it is accurate, clear, and complete, reply with exactly this one word: APPROVED. "
        "Otherwise give one or two specific changes. Do not say APPROVED unless you accept it.\n\n"
        f"Explanation:\n{state['draft']}"
    )
    critique = llm.invoke(prompt).content.strip()
    print(f"review: {critique}")
    return {"critique": critique}


def keep_going(state: ReflectState):
    if _accepted(state["critique"]) or state["iteration"] >= 3:
        return END
    return "generate"


reflect_graph = StateGraph(ReflectState)
reflect_graph.add_node("generate", generate)
reflect_graph.add_node("reflect", reflect)
reflect_graph.add_edge(START, "generate")
reflect_graph.add_edge("generate", "reflect")
reflect_graph.add_conditional_edges(
    "reflect", keep_going, {END: END, "generate": "generate"}
)
reflection_app = reflect_graph.compile()

print("Graph edges:")
for edge in reflection_app.get_graph().edges:
    kind = "conditional" if edge.conditional else "always"
    print(f"  {edge.source} --{kind}--> {edge.target}")
result = reflection_app.invoke({"task": "why we use the ReLU activation function"})
print(f"\nStopped after {result['iteration']} draft(s).")
print("ACCEPTED" if _accepted(result["critique"]) else "STOPPED AT THE CAP")


Graph edges:
  __start__ --always--> generate
  generate --always--> reflect
  reflect --conditional--> __end__
  reflect --conditional--> generate



--- draft 1 ---
ReLU introduces non-linearity into neural networks while allowing neurons to remain inactive (outputting zero) for negative inputs, which effectively prunes unused connections and simplifies computation without requiring complex mathematical derivatives.



review: The explanation is accurate regarding the core benefits of ReLU but lacks completeness by omitting its primary advantage over sigmoid/tanh: mitigating the vanishing gradient problem in deep networks during backpropagation, which allows for faster training convergence compared to other activation functions that saturate at negative values.



--- draft 2 ---
ReLU introduces non-linearity into neural networks while allowing neurons to remain inactive (outputting zero) for negative inputs, effectively pruning unused connections and simplifying computation without requiring complex mathematical derivatives. Crucially, unlike sigmoid or tanh functions that saturate at negative values causing gradients to vanish during backpropagation in deep networks, ReLU maintains a constant gradient of 1 for positive inputs. This property significantly mitigates the vanishing gradient problem, enabling faster training convergence and allowing models with many layers to be trained effectively. Consequently, these characteristics make ReLU one of the most widely used activation functions in modern deep learning architectures.



review: APPROVED

Stopped after 2 draft(s).
ACCEPTED


## When to use it

Use reflection when a weak first answer is more expensive than a few extra model calls: a public explanation, code that will be tested, a summary someone will file.

**You should see** a conditional edge from `reflect` back to `generate`, then a thin draft, a review that asks for a change, a second draft, and either `ACCEPTED` or `STOPPED AT THE CAP`. Three drafts is the ceiling in `keep_going`. The first prompt is deliberately thin so the loop is visible. Delete that restriction and a good first draft can be accepted immediately.

`reflection_app.invoke(..., debug=True)` prints every state update. Leave it off in class. It is a lot of text.

**Try it.** Change the reviewer prompt so it accepts a draft only when a new student could follow it. The later drafts should get shorter and more concrete.


---
# Pattern 4 — Plan-and-execute

## Theory

A ReAct agent picks only the next step. On a long task it can drop an earlier requirement. Plan-and-execute splits that into two nodes:

1. **Planner** — one structured call that returns an ordered list of steps.
2. **Executor** — one short call per step, in that order.

```
objective ──▶ planner ──▶ [step 1, step 2, step 3] ──▶ executor ──▶ result
```

You can read the plan before any step runs. The expensive "what is the whole job?" question is asked once.

A stronger version re-plans after the executor: are we done, or does the plan need a change? This notebook stays on one pass. The last section says how to add the second pass.


In [9]:
# ── Pattern 4: plan, then execute each step ───────────────────────────────────

class Plan(BaseModel):
    """An ordered list of concrete steps."""
    steps: list[str] = Field(description="exactly 3 concrete, ordered steps")


class PlanState(TypedDict):
    objective: str
    plan: list
    result: str


planner = llm.with_structured_output(Plan, method="function_calling")


def plan_node(state: PlanState):
    plan = planner.invoke(
        "Create a 3-step plan. Each step is one short instruction.\n\n"
        f"Objective: {state['objective']}"
    )
    return {"plan": plan.steps[:3]}


def execute_node(state: PlanState):
    done = []
    for number, step in enumerate(state["plan"], 1):
        reply = llm.invoke(
            f"Objective: {state['objective']}\n"
            "Do only this step, in 1 or 2 short sentences. Do not perform the other steps.\n"
            f"Step {number}: {step}"
        ).content.strip()
        done.append(f"Step {number}: {step}\n   -> {reply}")
    return {"result": "\n\n".join(done)}


plan_graph = StateGraph(PlanState)
plan_graph.add_node("plan", plan_node)
plan_graph.add_node("execute", execute_node)
plan_graph.add_edge(START, "plan")
plan_graph.add_edge("plan", "execute")
plan_graph.add_edge("execute", END)
planexec_app = plan_graph.compile()

out = planexec_app.invoke({
    "objective": (
        "Design a beginner mini-project that trains one decision tree "
        "on the iris dataset, which has 3 species and 4 measurements"
    )
})
print("PLAN")
for number, step in enumerate(out["plan"], 1):
    print(f"  {number}. {step}")
print("\nEXECUTION\n")
print(out["result"])


PLAN
  1. Load the Iris dataset containing sepal length/width and petal length/width for three flower species.
  2. Split the data into training and testing sets to evaluate model performance on unseen samples.
  3. Train a single decision tree classifier using only the four measurement features as inputs.

EXECUTION

Step 1: Load the Iris dataset containing sepal length/width and petal length/width for three flower species.
   -> Load the `iris` dataset from scikit-learn to access its features (sepal and petal lengths) and target labels (species). Split this data into training and testing sets using a simple train-test split function before initializing your decision tree model.

Step 2: Split the data into training and testing sets to evaluate model performance on unseen samples.
   -> Split the iris dataset randomly into a training set (e.g., 70%) for building the decision tree and a test set (30%) for evaluating its accuracy on unseen samples using scikit-learn's `train_test_split`

## When to use it

Use it when the steps depend on each other and you want the checklist visible. A comparison report is the usual case: name the three frameworks, compare them on one axis, then write the verdict. This demo asks for three steps because `Plan.steps` is capped at three.

**You should see** a three-step plan, then a short result under each step. The executor sometimes starts the next step early. That is the model ignoring its instruction. The graph still has one plan node and one execute node, and it does not add a fourth step.

**Try it.** Print the step number inside `execute_node` before the model call, then change the objective to *Plan a 5-day study schedule for an ML exam.* The schema still returns three steps until you change `Plan`.


---
# Pattern 5 — Human-in-the-loop

## Theory

An autonomous agent will eventually propose something you do not want sent, deleted, or paid. A human-in-the-loop gate pauses the graph, shows the proposed action, and waits.

```
prepare ──▶ interrupt ──▶ person decides ──▶ act, or cancel
```

`interrupt(payload)` raises inside the node and saves `payload` for your application. The checkpointer stores the state, so the process can wait. `Command(resume=decision)` continues **the same thread**. The node runs again from the top, and this time `interrupt(...)` returns the decision instead of pausing.

A checkpointer is required. `MemorySaver` is the in-process one. A server uses a database checkpointer so the pause survives a restart.


In [10]:
# ── Pattern 5: pause for approval ─────────────────────────────────────────────

class HITLState(TypedDict):
    request: str
    approved: str
    result: str


def approval_gate(state: HITLState):
    decision = interrupt({
        "action_to_approve": state["request"],
        "instructions": "Reply yes to proceed or no to cancel.",
    })
    return {"approved": decision}


def act(state: HITLState):
    if str(state["approved"]).strip().lower().startswith("y"):
        return {"result": f"PROCEEDED: {state['request']}"}
    return {"result": f"CANCELLED: {state['request']}"}


hitl_graph = StateGraph(HITLState)
hitl_graph.add_node("approval_gate", approval_gate)
hitl_graph.add_node("act", act)
hitl_graph.add_edge(START, "approval_gate")
hitl_graph.add_edge("approval_gate", "act")
hitl_graph.add_edge("act", END)

# A new thread each run. Reusing a finished or paused thread_id makes the
# second invoke fail or resume the old decision.
hitl_app = hitl_graph.compile(checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": f"approval-{uuid4().hex[:8]}"}}

paused = hitl_app.invoke(
    {"request": "DELETE the practice records labelled 'demo-batch'"},
    cfg,
)
print("PAUSED")
print("thread:", cfg["configurable"]["thread_id"])
print("waiting on:", hitl_app.get_state(cfg).next)
print("proposed:", paused["__interrupt__"][0].value["action_to_approve"])


PAUSED
thread: approval-f01e238b
waiting on: ('approval_gate',)
proposed: DELETE the practice records labelled 'demo-batch'


In [11]:
# ── Pattern 5: resume with the person's decision ──────────────────────────────
# Run the pause cell first. Run it again if you want a fresh proposal.
# Change human_decision to "yes" to take the other branch.

human_decision = "no"

if "hitl_app" not in globals() or "cfg" not in globals():
    print("Run the pause cell first.")
elif not hitl_app.get_state(cfg).next:
    print("This thread already finished. Run the pause cell again, then this cell.")
else:
    final = hitl_app.invoke(Command(resume=human_decision), cfg)
    print(f"Human said: {human_decision}")
    print(final["result"])


Human said: no
CANCELLED: DELETE the practice records labelled 'demo-batch'


## When to use it

Use it before an action that is expensive or hard to undo: sending mail, paying an invoice, deleting rows, stopping a machine.

The pause cell proposes a delete and stops. The resume cell answers `no`, so the graph takes the cancel branch. Nothing outside this process is deleted. The `result` string is the graph telling you which branch ran.

**You should see** `PAUSED`, a thread id, and `waiting on: ('approval_gate',)`. The next cell prints `CANCELLED`.

**Try it.** Run the pause cell again, set `human_decision = "yes"`, and run the resume cell. Then add a second interrupt later in the graph and resume twice.


---
# Pattern 6 — Supervisor

## Theory

One agent with many unrelated tools starts calling the wrong one. Split the work. Specialists do one job. A supervisor reads the conversation and chooses who speaks next.

```
                 supervisor
                /          \
        math_expert      ml_tutor
                \          /
                 supervisor
```

The supervisor returns `Command(goto=...)`, which jumps to a node even when there is no static edge. Pass `destinations` so the graph knows the legal jumps and a typo fails at compile time.

The route itself is a schema (`math_expert`, `ml_tutor`, or `FINISH`), using the same `function_calling` method as Pattern 2. A step counter forces the loop to end. Specialists report back to the supervisor, who then finishes or routes again.


In [12]:
# ── Pattern 6: supervisor and two specialists ─────────────────────────────────

class SupervisorState(MessagesState):
    steps: int


class Route(BaseModel):
    """Who should act next."""
    next: Literal["math_expert", "ml_tutor", "FINISH"]


def _render(messages):
    lines = []
    for message in messages:
        role = type(message).__name__.replace("Message", "")
        name = getattr(message, "name", None)
        who = f"{role}/{name}" if name else role
        lines.append(f"{who}: {message.content[:180]}")
    return "\n".join(lines)


def supervisor(state: SupervisorState) -> Command:
    steps = state.get("steps", 0)
    if steps >= 4:
        return Command(goto=END, update={"steps": steps})
    decision = llm.with_structured_output(Route, method="function_calling").invoke(
        "You route a request to one specialist.\n"
        "- math_expert: arithmetic\n"
        "- ml_tutor: explains a machine-learning idea in words\n"
        "Pick who acts NEXT. Pick FINISH when a specialist has already answered the user.\n\n"
        f"Conversation:\n{_render(state['messages'])}"
    )
    nxt = END if decision.next == "FINISH" else decision.next
    print(f"   supervisor step {steps} -> {decision.next}")
    return Command(goto=nxt, update={"steps": steps + 1})


def make_specialist(name: str, system_prompt: str):
    def node(state: SupervisorState) -> Command:
        response = llm.invoke([SystemMessage(system_prompt), *state["messages"]])
        text = response.content.strip() or "(empty reply)"
        print(f"   {name} replied")
        return Command(
            goto="supervisor",
            update={"messages": [AIMessage(content=text, name=name)]},
        )
    return node


super_graph = StateGraph(SupervisorState)
super_graph.add_node(
    "supervisor", supervisor, destinations=("math_expert", "ml_tutor", END)
)
super_graph.add_node(
    "math_expert",
    make_specialist(
        "math_expert",
        "You are a math expert. Give the exact numeric result in one sentence.",
    ),
    destinations=("supervisor",),
)
super_graph.add_node(
    "ml_tutor",
    make_specialist(
        "ml_tutor",
        "You are an ML tutor. Explain the idea in two sentences.",
    ),
    destinations=("supervisor",),
)
super_graph.add_edge(START, "supervisor")
supervisor_app = super_graph.compile()

print("Q: What is 25 times 4?\n")
res = supervisor_app.invoke(
    {"messages": [HumanMessage("What is 25 times 4?")], "steps": 0}
)
print("\nFinal:", res["messages"][-1].content)


Q: What is 25 times 4?



   supervisor step 0 -> math_expert


   math_expert replied


   supervisor step 1 -> FINISH

Final: The product of 25 and 4 is exactly 100.


## When to use it

Use a supervisor when one request crosses skills that want different prompts or different tools. A banking assistant can send a transfer to a specialist that has payment tools and an approval gate, and send "what is compound interest?" to a tutor. You can replace either specialist without rewriting the other.

**You should see** `supervisor step 0 -> math_expert`, a reply, then `FINISH`. The final line is the math specialist's message.

The math specialist is still a prompted model. It is not the calculator from Pattern 1. For a number that must be exact, make that specialist its own `create_agent` with the `calculator` tool, and return only its final message to the supervisor.

**Try it.** Add a `code_helper` destination that writes a three-line Python snippet. Ask: *Explain a for-loop, and what is 3 + 5?* The supervisor should visit two specialists before `FINISH`.


---
# Pattern 7 — Long-term memory

## Theory

LangGraph has two stores, and they are easy to mix up.

| | Checkpointer | Store |
|---|---|---|
| Object here | `MemorySaver` | `InMemoryStore` |
| What it holds | The messages of one thread | Facts you choose to write |
| Key | `thread_id` | A namespace, here `("memories", "user_priya")` |
| New thread | The chat is empty | The facts are still there |
| Kernel restart | Gone | Gone |

Pattern 5 used the checkpointer so a pause can resume. This pattern uses the store so a **new** conversation still knows a fact from the last one. A production store is Redis or Postgres. `InMemoryStore` is only alive in this kernel.

One turn does three things: read the user's facts, answer with those facts in the prompt, then write any new fact from this message.


In [13]:
# ── Pattern 7: remember a fact, then start a new chat ─────────────────────────

store = InMemoryStore()
USER = ("memories", "user_priya")  # (bucket, user id)

# Running this cell again should not stack duplicate facts.
for item in list(store.search(USER)):
    store.delete(USER, item.key)


class Fact(BaseModel):
    """A durable fact about the user, or none."""
    has_fact: bool = Field(description="true only if the message states a personal fact")
    fact: str = Field(description="the fact in third person, such as 'is preparing for an ML exam'")


extractor = llm.with_structured_output(Fact, method="function_calling")


def chat_with_memory(user_text: str) -> str:
    known = [item.value["text"] for item in store.search(USER)]
    context = "\n".join(f"- {fact}" for fact in known) if known else "(nothing yet)"
    answer = llm.invoke(
        f"Known facts about the user:\n{context}\n\n"
        f"User says: {user_text}\n\n"
        "Reply in 2 sentences. Use a fact when it is relevant."
    ).content.strip()

    found = extractor.invoke(
        "Does this message contain a personal fact to remember? "
        f"Message: {user_text}"
    )
    fact = (found.fact or "").strip()
    if found.has_fact and fact.lower() not in {"", "none", "n/a"}:
        store.put(USER, str(uuid4()), {"text": fact})
        print(f"   saved: {fact}")
    return answer


print("SESSION 1")
print(chat_with_memory("Hi! I'm Priya and I'm preparing for my ML exam next week."))
print()
print(chat_with_memory("I find decision trees really confusing."))


SESSION 1


   saved: is preparing for an ML exam
Hi Priya, good luck with your machine learning exam next week! While many students focus on algorithms like decision trees or neural networks, remember that understanding the underlying mathematical principles of optimization often provides deeper insight than just memorizing code snippets.



   saved: finds decision trees really confusing
Decision trees are often difficult because they rely on splitting data based on specific rules that can become overly complex if not pruned correctly, leading to overfitting where the model memorizes noise instead of learning patterns. A key fact about their structure is that each internal node represents a feature test while every leaf node holds a class label or decision outcome.


In [14]:
# ── Pattern 7: a new session, same store ──────────────────────────────────────
# There is no message history here. The store is the only memory.

print("Stored facts:")
for item in store.search(USER):
    print(f"  - {item.value['text']}")

print("\nSESSION 2")
print(chat_with_memory("Hey, I'm back! What was I working on?"))


Stored facts:
  - is preparing for an ML exam
  - finds decision trees really confusing

SESSION 2


You were diving into decision trees to prepare for your upcoming machine learning exam. Since you mentioned finding them particularly confusing earlier, we can focus specifically on how they split data using entropy or information gain if that helps clarify things now.


## When to use it

Use the store when the assistant should still know the user next week. A coding tutor that remembers "knows Python, finds recursion hard" can use that fact in a later thread about trees.

**You should see** one or two `saved:` lines in session 1, the same facts listed at the start of session 2, and a reply that mentions the exam or decision trees. Session 2 does not receive the session 1 messages. If the reply ignores the facts, look at the stored list before you blame the prompt.

Restarting the kernel clears `InMemoryStore`. That surprises people who just learned that it survives a new thread.

**Try it.** In session 2, ask *Which topic should I review?* The reply should name decision trees if that fact was saved.


---
# Pattern 8 — Agentic RAG

## Theory

Classic RAG retrieves on every question, including "hello". Agentic RAG makes retrieval a tool. The agent calls it only when the question depends on private facts.

```
question ──▶ agent ──▶ need the knowledge base?
                         │ yes              │ no
                         ▼                  ▼
                    search tool      answer from the model
                         │
                         ▼
                    answer using the tool result
```

This cell uses a dictionary, not Chroma. The lesson is the decision. Notebooks 2 and 3 in this folder are the document version: a real index, a grade step, and a person in the loop before a measured number is treated as sent.

The full pattern also grades the retrieved text and rewrites the query when it is a miss. That second loop is Pattern 3 wrapped around a retriever.


In [15]:
# ── Pattern 8: retrieve only for course facts ─────────────────────────────────
# Toy facts for this notebook. They are not the real course calendar.

COURSE_KB = {
    "exam date": "The practice quiz for this notebook is on 15 April. This is toy data.",
    "instructor": "This demo is taught by the LogicMojo instructor.",
    "passing mark": "A score of 60% is required to pass the practice certification in this notebook.",
}


@tool
def search_course_kb(query: str) -> str:
    """Search this notebook's course facts: exam date, instructor, passing mark.
    Use it for questions about this course. Do not use it for general ML ideas.
    """
    query_l = query.lower()
    hits = [
        value
        for key, value in COURSE_KB.items()
        if any(word in query_l for word in key.split())
    ]
    return "\n".join(hits) if hits else "No course fact matched that query."


rag_agent = create_agent(
    model=llm,
    tools=[search_course_kb],
    system_prompt=(
        "You are a course assistant. For the exam, the instructor, or the passing "
        "mark, call search_course_kb. For a general ML idea, answer in one sentence "
        "and do not call the tool."
    ),
)


def ask_rag(question: str):
    result = rag_agent.invoke({"messages": [HumanMessage(question)]})
    retrieved = any(getattr(message, "tool_calls", None) for message in result["messages"])
    print(f"Q: {question}")
    print("   path:", "retrieved" if retrieved else "answered without retrieval")
    print(f"   A: {result['messages'][-1].content}\n")


ask_rag("When is our ML final exam?")
ask_rag("In one sentence, what is a neural network?")


Q: When is our ML final exam?
   path: retrieved
   A: Based on the course information, our ML final exam will be held on **April 15**. Please note that this refers to a practice or sample date mentioned in the available notes; you should verify with your instructor for the official schedule of any upcoming exams.



Q: In one sentence, what is a neural network?
   path: answered without retrieval
   A: A neural network is an artificial intelligence model inspired by the human brain that learns to recognize patterns and make predictions through interconnected layers of nodes processing data.



## When to use it

Use it when some questions must come from your documents and others should not. "What is our refund policy?" needs the policy. "What is the capital of France?" does not.

**You should see** `retrieved` for the exam question and `answered without retrieval` for the neural-network question. The exam answer should mention 15 April, from the dictionary. If both questions retrieve, the system prompt is too broad. If neither retrieves, the tool docstring is too easy to ignore.

The dictionary match is literal: the tool query has to contain a word from the key, such as `exam` or `instructor`. A vector index, as in notebook 2, is what you use when the wording will vary.

**Try it.** Add `"office hours": "Office hours in this demo are Thursday at 4pm."` and ask *When are office hours?* then *What is gradient descent?* Only the first should retrieve.


---
## Pitfalls

1. **Schema method.** `with_structured_output(Schema)` defaults to `json_schema`. On `qwen3.5:4b` pass `method="function_calling"`, or the parser raises `OutputParserException` on a markdown reply.
2. **The word APPROVED.** `"APPROVED" in critique` is true for `NOT APPROVED`. Check the first line, as `_accepted` does.
3. **Resume the paused thread.** `interrupt` needs a checkpointer. Resume with `Command(resume=...)` on that `thread_id`. A new human message does not answer the pause. Run the pause cell again when you want a new proposal.
4. **Chat memory and user memory.** A new `thread_id` empties the chat and leaves the store. Restarting the kernel empties both. Production replaces `InMemoryStore` and `MemorySaver` with a database.
5. **Skipped tools.** A 4B model skips a tool when the description is vague. Say when the tool is required, in the docstring and in the system prompt.

## Extension

Add a `replan` node after `execute` in Pattern 4. Give it the objective and the execution text. If the result does not meet the objective, replace `plan` and run the executor once more. Cap it at two plans so it cannot loop.
